# Análise Textual de Contratos PNCP

Sequência didática aplicada aos contratos do PNCP:

1. Pré-processamento de textos + Bag-of-Words/TF-IDF (1,2,3-gramas)
2. Rede k-NN sobre TF-IDF + agrupamento por Label Propagation
3. Descritores por cluster (termos de maior TF-IDF)
4. Embeddings contextuais (Sentence-BERT multilíngue)
5. Rede k-NN sobre embeddings + clusters
6. Geração de indicadores por cluster via LLM (Google Gemini)
7. Classificação semissupervisionada por regularização em grafo
8. Agente LLM com ferramenta de busca semântica

Pré-requisito: ter o arquivo `contratos.parquet` no Google Drive
(produzido por uma coleta anterior).

## Bibliotecas

In [ ]:
!pip install -q sentence-transformers networkx plotly nltk
!pip install -q google-generativeai
!pip install -q pyarrow pandas

In [ ]:
import pandas as pd
import numpy as np

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics.pairwise import cosine_similarity

import networkx as nx
from networkx.algorithms import community

import plotly.graph_objects as go

## Métodos de apoio

- `meu_tokenizador`: caixa-baixa + tokenização + remove stopwords (PT-BR) + radicalização RSLP.
- `get_cluster_descriptors`: para um cluster, devolve os termos com maior soma TF-IDF.
- `show_graph_cluster`/`show_graph_regularization`: visualização interativa Plotly.
- `simple_regularization`: classificação semissupervisionada por propagação iterativa sobre o grafo.

In [ ]:
STOP_PT = set(nltk.corpus.stopwords.words('portuguese'))
STEMMER = RSLPStemmer()

def remove_stopwords(texto, stop_words=STOP_PT):
    s = str(texto).lower()
    tokens = word_tokenize(s, language='portuguese')
    return [w for w in tokens
            if w not in stop_words and w.isalnum() and not w.isdigit() and len(w) > 2]

def stemming(tokens, stemmer=STEMMER):
    return [stemmer.stem(w) for w in tokens]

def meu_tokenizador(doc):
    return stemming(remove_stopwords(doc))

def get_cluster_descriptors(VSM, df_documentos, cluster_id, max_terms=5):
    sub = df_documentos[df_documentos.cluster == cluster_id]['text']
    df_desc = pd.DataFrame({
        'word': VSM.get_feature_names_out(),
        'tfidf_sum': VSM.transform(sub).toarray().sum(axis=0),
    }).sort_values('tfidf_sum', ascending=False)
    return len(sub), df_desc[df_desc.tfidf_sum > 0].head(max_terms).word.tolist()

In [ ]:
def _arestas_xy(G):
    ex, ey = [], []
    for u, v in G.edges():
        x0, y0 = G.nodes[u]['pos']
        x1, y1 = G.nodes[v]['pos']
        ex += [x0, x1, None]
        ey += [y0, y1, None]
    return ex, ey

def show_graph_cluster(G):
    ex, ey = _arestas_xy(G)
    edge_trace = go.Scatter(x=ex, y=ey,
                            line=dict(width=2, color='#888'),
                            hoverinfo='none', mode='lines')
    nx_, ny_, txt, cor = [], [], [], []
    for n in G.nodes():
        x, y = G.nodes[n]['pos']
        nx_.append(x); ny_.append(y)
        txt.append(G.nodes[n].get('text', str(n)))
        cor.append(G.nodes[n].get('cluster', 0))
    node_trace = go.Scatter(x=nx_, y=ny_, mode='markers', hoverinfo='text',
                            text=txt,
                            marker=dict(size=10, line_width=2, color=cor))
    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(showlegend=False, hovermode='closest',
                                     margin=dict(b=20, l=5, r=5, t=40),
                                     xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                     yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    fig.show()

def show_graph_regularization(G):
    ex, ey = _arestas_xy(G)
    edge_trace = go.Scatter(x=ex, y=ey, line=dict(width=1, color='#888'),
                            hoverinfo='none', mode='lines')
    nx_, ny_, txt, cor = [], [], [], []
    for n in G.nodes():
        x, y = G.nodes[n]['pos']
        nx_.append(x); ny_.append(y)
        f = G.nodes[n].get('f', np.array([0.0]))
        txt.append(f'{f}<br>{G.nodes[n].get("text", str(n))}')
        cor.append(float(f[0]))
    node_trace = go.Scatter(x=nx_, y=ny_, mode='markers', hoverinfo='text',
                            text=txt,
                            marker=dict(showscale=True, colorscale='Reds',
                                        size=10, line_width=2, color=cor))
    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(showlegend=False, hovermode='closest',
                                     margin=dict(b=20, l=5, r=5, t=40),
                                     xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                     yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    fig.show()

In [ ]:
def simple_regularization(G, labels, max_iter=30):
    """Propagação iterativa: cada nó recebe a média dos vizinhos; nós
    rotulados (seeds) ficam fixos. Resultado em G.nodes[n]['f']."""
    for n in G.nodes():
        G.nodes[n]['f'] = np.array([0.0])
        if n in labels:
            G.nodes[n]['y'] = np.array([1.0])
            G.nodes[n]['f'] = np.array([1.0])
    for it in range(max_iter):
        diff = 0.0
        for node in G.nodes():
            if node in labels:
                continue
            viz = list(G.neighbors(node))
            if not viz:
                continue
            f_new = np.mean([G.nodes[v]['f'] for v in viz], axis=0)
            diff += float(np.linalg.norm(G.nodes[node]['f'] - f_new))
            G.nodes[node]['f'] = f_new
            if 'y' in G.nodes[node]:
                G.nodes[node]['f'] = G.nodes[node]['y']
        print(f'Iteração #{it+1} Q(F)={diff:.4f}')

## Modelo LLM (Google Gemini)

Configura a API do Gemini lendo a chave do `userdata` do Colab.

**Como configurar a chave:**
1. No Colab, clique no ícone de chave 🔑 (Secrets) na barra lateral esquerda
2. Adicione um secret chamado `GOOGLE_API_KEY` com sua chave da AI Studio
3. Marque "Notebook access" para que ele apareça via `userdata.get(...)`

Pegue uma chave gratuita em https://aistudio.google.com/app/apikey

In [ ]:
import google.generativeai as genai
from google.colab import userdata

def llm_local():
    GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print('Gemini API configured.')

def llm_task(model, system, prompt):
    gemini_model = genai.GenerativeModel(model_name=model)
    response = gemini_model.generate_content(
        [system, prompt]
    )
    return response.text

llm_local()


## Base textual — contratos PNCP

Lê o consolidado `contratos.parquet` diretamente do Drive. **Não precisa**
do pacote `pncp` — basta o arquivo coletado previamente.

Para a parte exploratória usamos uma amostra estratificada de até 3000 contratos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Ajuste o caminho conforme onde você guardou os dados no Drive
CAMINHO_PARQUET = '/content/drive/MyDrive/PNCP_TCC/dados/coleta/contratos.parquet'

df_contratos = pd.read_parquet(
    CAMINHO_PARQUET,
    columns=['numeroControlePNCP', 'objeto', 'rotulo',
             'razaoSocialOrgao', 'municipioNome', 'valor'],
)
df_contratos = df_contratos.rename(columns={'objeto': 'text'})
df_contratos = df_contratos.dropna(subset=['text'])
df_contratos = df_contratos[df_contratos['text'].str.len() > 20].reset_index(drop=True)

# Amostra estratificada por rótulo (até 3000 contratos)
ALVO = 3000
if len(df_contratos) > ALVO:
    df_amostra = (df_contratos.groupby('rotulo', group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), ALVO // 3),
                                              random_state=42)))
    df_amostra = df_amostra.reset_index(drop=True)
else:
    df_amostra = df_contratos.reset_index(drop=True)

print(f'Total na base: {len(df_contratos):,} | amostra desta análise: {len(df_amostra):,}')
df_amostra.head()

Mounted at /content/drive


## Modelo Espaço-Vetorial + TF-IDF

TF-IDF com tokenizador customizado (stopwords PT + stemmer RSLP), corte por
frequência mínima de documentos (`min_df`).

In [ ]:
VSM = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None, min_df=3)
X = VSM.fit_transform(df_amostra['text'])
print(f'X: {X.shape[0]:,} docs x {X.shape[1]:,} termos (nnz={X.nnz:,})')

df_word_tfidfs = pd.DataFrame({
    'word': VSM.get_feature_names_out(),
    'tfidf_sum': X.toarray().sum(axis=0),
}).sort_values('tfidf_sum', ascending=False)
df_word_tfidfs.head(50)

## N-gramas (bigramas e trigramas)

Mostra termos compostos relevantes (ex.: *projeto básico*, *obra civil*, *manutenção predial*).

In [ ]:
VSM_bi = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                          min_df=3, ngram_range=(2, 2))
X_bi = VSM_bi.fit_transform(df_amostra['text'])
df_bi = pd.DataFrame({
    'word': VSM_bi.get_feature_names_out(),
    'tfidf_sum': X_bi.toarray().sum(axis=0),
}).sort_values('tfidf_sum', ascending=False)
df_bi.head(30)

In [ ]:
VSM_tri = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                           min_df=3, ngram_range=(3, 3))
X_tri = VSM_tri.fit_transform(df_amostra['text'])
df_tri = pd.DataFrame({
    'word': VSM_tri.get_feature_names_out(),
    'tfidf_sum': X_tri.toarray().sum(axis=0),
}).sort_values('tfidf_sum', ascending=False)
df_tri.head(30)

## Rede k-NN sobre TF-IDF (1+2-gramas) + Label Propagation

In [ ]:
VSM = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                       min_df=3, ngram_range=(1, 2))
X = VSM.fit_transform(df_amostra['text'])
A = kneighbors_graph(X, n_neighbors=5, metric='cosine')
G = nx.Graph(A)

cluster_id = 0
for clusters in community.label_propagation_communities(G):
    for doc_id in clusters:
        G.nodes[doc_id]['cluster'] = cluster_id
    cluster_id += 1

df_amostra['cluster'] = [G.nodes[i].get('cluster', -1) for i in df_amostra.index]
df_amostra.head()

In [ ]:
# Estatísticas por cluster
df_stats = (df_amostra[['text', 'cluster']].groupby('cluster').count()
            .sort_values(by='text', ascending=False)
            .rename(columns={'text': 'num_docs'}))
df_stats.head(20)

### Descritores de cluster (termos com maior TF-IDF)

In [ ]:
QTD_TOPICOS = 15
linhas = []
for c in df_stats.head(QTD_TOPICOS).index:
    n_docs, descs = get_cluster_descriptors(VSM, df_amostra, c)
    linhas.append([c, n_docs, descs])
df_descriptors = pd.DataFrame(linhas, columns=['cluster', 'num_docs', 'descriptors'])
df_descriptors

### Subgrafo dos top-N clusters + visualização

In [ ]:
selected = df_descriptors.cluster.tolist()
G2 = G.copy()
for n in list(G2.nodes()):
    if G2.nodes[n].get('cluster') not in selected:
        G2.remove_node(n)

pos = nx.spring_layout(G2, seed=42)
for n in G2.nodes():
    G2.nodes[n]['pos'] = pos[n]

for i, row in df_amostra.iterrows():
    if i in G2.nodes:
        descs = df_descriptors[df_descriptors.cluster == G2.nodes[i]['cluster']]
        cabec = str(descs.descriptors.tolist()[0]) if not descs.empty else ''
        G2.nodes[i]['text'] = (str(row.get('rotulo', '')) + ' | ' + cabec +
                                  '<br>' + str(row['text'])[:120])

show_graph_cluster(G2)

## Embeddings contextuais (Sentence-BERT multilíngue)

DistilBERT multilíngue ajustado para similaridade semântica entre sentenças.

In [ ]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('distiluse-base-multilingual-cased-v1')
X_emb = sbert.encode(df_amostra['text'].tolist(), show_progress_bar=True,
                       batch_size=64, convert_to_numpy=True,
                       normalize_embeddings=True)
X_emb.shape

In [ ]:
A_emb = kneighbors_graph(X_emb, n_neighbors=5, metric='cosine')
G_emb = nx.Graph(A_emb)

cid = 0
for clusters in community.label_propagation_communities(G_emb):
    for doc_id in clusters:
        G_emb.nodes[doc_id]['cluster'] = cid
    cid += 1
df_amostra['cluster_sbert'] = [G_emb.nodes[i].get('cluster', -1)
                                  for i in df_amostra.index]

df_stats_emb = (df_amostra[['text', 'cluster_sbert']]
                .groupby('cluster_sbert').count()
                .sort_values('text', ascending=False)
                .rename(columns={'text': 'num_docs'}))
df_stats_emb.head(20)

### Subgrafo dos top-50 clusters semânticos + visualização

In [ ]:
selected_emb = list(df_stats_emb.head(50).index)
G_emb2 = G_emb.copy()
for n in list(G_emb2.nodes()):
    if G_emb2.nodes[n].get('cluster') not in selected_emb:
        G_emb2.remove_node(n)

pos = nx.spring_layout(G_emb2, seed=42)
for n in G_emb2.nodes():
    G_emb2.nodes[n]['pos'] = pos[n]

for i, row in df_amostra.iterrows():
    if i in G_emb2.nodes:
        G_emb2.nodes[i]['text'] = (str(row.get('rotulo', '')) + '<br>' +
                                       str(row['text'])[:120])

show_graph_cluster(G_emb2)

## Geração de indicadores por cluster via LLM

Para cada cluster grande, o modelo lê uma amostra dos objetos e devolve um
indicador JSON estruturado: nome, categoria e resumo.

In [ ]:
SYSTEM_INDICADOR = '''
Você é analista de inteligência analítica especializado em contratações
públicas brasileiras (Lei 14.133/2021).

Recebe uma amostra de objetos de contratos agrupados por similaridade
semântica (cluster). Sua tarefa: identificar UM indicador-chave que
sintetize o padrão presente no cluster, especialmente quando houver indício
de subenquadramento (contratos rotulados como "serviços gerais" que
deveriam ser "engenharia/obras").

Responda APENAS no JSON:
{
  "indicador": {
    "nome": "Nome curto e descritivo",
    "categoria": "obra civil | manutenção | serviço técnico de engenharia | vigilância/limpeza | fornecimento | transporte | outro",
    "resumo": "1-2 frases sobre o padrão observado e por que importa para auditoria/controle",
    "indicio_subenquadramento": "alto | medio | baixo | nenhum",
    "recomendacao": "uma ação concreta"
  }
}
'''

In [ ]:
TOP_N_CLUSTERS = 10
N_OBJETOS_POR_CLUSTER = 8
# Modelos Gemini disponíveis: gemini-1.5-flash (rápido/barato),
# gemini-1.5-pro (qualidade máxima), gemini-2.0-flash-exp (mais recente).
# 'gemini-pro' antigo foi descontinuado — use os modelos acima.
MODELO_LLM = 'gemini-1.5-flash'

indicadores = []
for cid in df_stats_emb.head(TOP_N_CLUSTERS).index:
    sub = df_amostra[df_amostra.cluster_sbert == cid].head(N_OBJETOS_POR_CLUSTER)
    objetos = '\n'.join(f'- {str(o)[:300]}' for o in sub['text'].tolist())
    prompt = (f'Cluster #{cid} ({len(sub)} objetos amostrados).\n\n'
              f'Objetos:\n{objetos}\n\nGere o indicador no JSON.')
    try:
        resp = llm_task(MODELO_LLM, SYSTEM_INDICADOR, prompt)
    except Exception as e:
        print(f'[cluster {cid}] LLM falhou: {e}')
        continue
    print(f'\n========= Cluster #{cid} (n={len(sub)}) =========')
    print(resp)
    indicadores.append({'cluster': cid, 'n_docs': len(sub), 'resposta': resp})

## Classificação semissupervisionada por regularização em grafo

Dado um pequeno conjunto de seeds confirmados como subenquadramento de
engenharia (ex.: contratos rotulados 'geral' contendo termos óbvios como
*ART*, *obra*, *projeto básico*), propagamos a "confiança" pelos vizinhos
do grafo k-NN. O resultado em `G.nodes[n]['f']` é um score 0–1.

In [ ]:
TERMOS_OBVIOS = ('art ', 'rrt ', 'crea', 'projeto básico', 'projeto basico',
                  'obra de engenharia', 'reforma predial', 'pavimentação',
                  'pavimentacao', 'memorial descritivo', 'abnt nbr')

labels = {}
for n in G_emb2.nodes():
    objeto = str(df_amostra.loc[n, 'text']).lower()
    rotulo = str(df_amostra.loc[n, 'rotulo'])
    if rotulo == 'geral' and any(t in objeto for t in TERMOS_OBVIOS):
        labels[n] = 1

print(f'Seeds encontradas (geral + termo óbvio): {len(labels)}')
for n in list(labels)[:5]:
    print(f'  #{n}: {str(df_amostra.loc[n, "text"])[:120]}')

In [ ]:
if labels:
    simple_regularization(G_emb2, labels, max_iter=30)
    show_graph_regularization(G_emb2)
else:
    print('Sem seeds — refine TERMOS_OBVIOS ou aumente a amostra.')

### Filtrando candidatos por limiar

In [ ]:
LIMIAR = 0.5
candidatos = []
for n in G_emb2.nodes():
    f = G_emb2.nodes[n].get('f', np.array([0.0]))
    if float(f[0]) > LIMIAR and n not in labels:
        candidatos.append(n)

df_candidatos = df_amostra.loc[candidatos, ['numeroControlePNCP', 'rotulo',
                                              'text', 'municipioNome', 'valor']]
print(f'Candidatos com score > {LIMIAR}: {len(df_candidatos)}')
df_candidatos.head(20)

## 9. Detecção de subenquadramento usando os rótulos limpos

**Hipótese metodológica:** os rótulos `engenharia` e `obras` têm alta precisão
(quando o órgão marcou assim, é porque é) — mas o rótulo `geral` é ruidoso:
parte é serviço comum de verdade, parte é engenharia mal classificada.

**Objetivo desta seção:** ranquear os contratos `geral` pela probabilidade de
serem na verdade engenharia/obras, usando 3 sinais complementares:

1. **Similaridade ao centróide** das embeddings de `engenharia + obras`
2. **Voto k-NN** — qual é a maioria dos vizinhos no grafo semântico?
3. **Pureza do cluster** — clusters dominados por engenharia/obras

> Nota: nesta etapa identificamos apenas a **rotulação incorreta**. A
> verificação posterior — se o processo seguiu o rito de engenharia (ART/RRT,
> projeto básico, etc.) — fica para uma etapa futura sobre os PDFs.

### 9.1 Centróide das classes "limpas" (engenharia + obras)

Os contratos rotulados "engenharia" e "obras" são razoavelmente confiáveis
(órgão acertou o cadastro). Calculamos o centróide das suas embeddings — o
"vetor médio" do que é engenharia — e medimos para cada "geral" a similaridade
ao centróide. Os mais próximos são candidatos fortes a estarem mal rotulados.

**Setup PU (Positive-Unlabeled)**: tratamos `engenharia ∪ obras` como
positivos limpos, e `geral` como conjunto misto (verdadeiros negativos +
falsos negativos que queremos encontrar).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
mask_pos = df_amostra['rotulo'].isin(['engenharia', 'obras'])
mask_geral = df_amostra['rotulo'] == 'geral'

print(f'Positivos limpos (engenharia + obras): {mask_pos.sum()}')
print(f'Não-rotulados a investigar (geral):    {mask_geral.sum()}')

# Centróide das embeddings positivas (vetor médio do "que é engenharia/obras")
centroide_pos = X_emb[mask_pos.values].mean(axis=0)
centroide_pos = centroide_pos / (np.linalg.norm(centroide_pos) + 1e-12)

# Similaridade cosseno de cada "geral" ao centróide positivo
sims = cosine_similarity(X_emb[mask_geral.values],
                          centroide_pos.reshape(1, -1)).ravel()

df_geral = df_amostra[mask_geral].copy().reset_index(drop=True)
df_geral['sim_centroide_eng'] = sims

# Distribuição da similaridade
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(sims, bins=60, color='#1f77b4')
ax.axvline(np.quantile(sims, 0.95), color='#d62728', linestyle='--',
            label='top 5% (percentil 95)')
ax.set_xlabel('similaridade ao centróide engenharia+obras')
ax.set_ylabel('nº de contratos "geral"')
ax.legend(); ax.set_title('Distância de cada "geral" ao "centro" de engenharia/obras')
plt.show()

print('\nTop 15 "geral" mais próximos do centróide eng/obras:')
df_geral.sort_values('sim_centroide_eng', ascending=False).head(15)[
    ['numeroControlePNCP', 'text', 'sim_centroide_eng']]

### 9.2 Voto k-NN — vizinhos com rótulo confiável

Para cada contrato "geral" no grafo k-NN do SBERT, olhamos os rótulos dos
vizinhos. Se a maioria dos vizinhos é engenharia/obras, este "geral" provavelmente
foi mal rotulado.

In [ ]:
votos = []
for n in G_emb.nodes():
    if n >= len(df_amostra):
        continue
    rotulo_n = df_amostra.iloc[n]['rotulo']
    if rotulo_n != 'geral':
        continue
    vizinhos = list(G_emb.neighbors(n))
    vizinhos_validos = [v for v in vizinhos if v < len(df_amostra)]
    if not vizinhos_validos:
        continue
    rot_viz = df_amostra.iloc[vizinhos_validos]['rotulo'].value_counts()
    n_eng = int(rot_viz.get('engenharia', 0) + rot_viz.get('obras', 0))
    n_geral_viz = int(rot_viz.get('geral', 0))
    n_total = n_eng + n_geral_viz
    if n_total == 0:
        continue
    votos.append({
        'numeroControlePNCP': df_amostra.iloc[n]['numeroControlePNCP'],
        'objeto': str(df_amostra.iloc[n]['text'])[:120],
        'n_vizinhos_eng_obras': n_eng,
        'n_vizinhos_geral': n_geral_viz,
        'pct_vizinhos_eng_obras': n_eng / n_total,
        'n_vizinhos': n_total,
    })
df_votos = (pd.DataFrame(votos)
            .sort_values('pct_vizinhos_eng_obras', ascending=False)
            .reset_index(drop=True))
print(f'{len(df_votos)} contratos "geral" com vizinhos analisados')
print(f'Com ≥50% dos vizinhos eng/obras: '
      f'{(df_votos["pct_vizinhos_eng_obras"] >= 0.5).sum()}')
df_votos.head(15)

### 9.3 Pureza de cluster — clusters dominados por engenharia/obras

Para cada cluster do SBERT, vemos a distribuição dos rótulos. Cluster com
predominância de engenharia/obras + alguns "geral" no meio → os "geral"
daquele cluster são candidatos fortes a subenquadramento.

In [ ]:
pureza = (df_amostra.groupby('cluster_sbert')['rotulo']
          .value_counts(normalize=False).unstack(fill_value=0))
for col in ('engenharia', 'obras', 'geral'):
    if col not in pureza.columns:
        pureza[col] = 0
pureza['total'] = pureza[['engenharia', 'obras', 'geral']].sum(axis=1)
pureza['pct_eng_obras'] = (
    (pureza['engenharia'] + pureza['obras']) / pureza['total'].clip(lower=1)
)

# Cluster suspeito: ≥60% engenharia/obras + ≥1 "geral" no meio + ≥5 contratos
clusters_suspeitos = pureza[
    (pureza['pct_eng_obras'] >= 0.6)
    & (pureza['geral'] >= 1)
    & (pureza['total'] >= 5)
].sort_values('pct_eng_obras', ascending=False)

print(f'Clusters suspeitos (eng/obras dominam, mas têm "geral" dentro): '
      f'{len(clusters_suspeitos)}')
clusters_suspeitos[['engenharia', 'obras', 'geral', 'total',
                     'pct_eng_obras']].head(20)

### 9.4 Consolidação — ranking por nº de sinais positivos

Os três sinais (centróide, voto kNN, cluster suspeito) são complementares.
Quanto mais sinais para um "geral", maior a chance de ele ser realmente
engenharia/obras mal rotulado.

In [ ]:
LIMIAR_CENTROIDE = df_geral['sim_centroide_eng'].quantile(0.85)  # top 15%
LIMIAR_VOTO = 0.5

df_ranking = df_geral.merge(
    df_votos[['numeroControlePNCP', 'pct_vizinhos_eng_obras', 'n_vizinhos']],
    on='numeroControlePNCP', how='left'
)
df_ranking['no_cluster_suspeito'] = df_ranking['cluster_sbert'].isin(
    clusters_suspeitos.index)

df_ranking['sinal_centroide'] = (
    df_ranking['sim_centroide_eng'] >= LIMIAR_CENTROIDE).astype(int)
df_ranking['sinal_voto_knn'] = (
    df_ranking['pct_vizinhos_eng_obras'].fillna(0) >= LIMIAR_VOTO).astype(int)
df_ranking['sinal_cluster'] = df_ranking['no_cluster_suspeito'].astype(int)

df_ranking['n_sinais'] = (df_ranking['sinal_centroide']
                            + df_ranking['sinal_voto_knn']
                            + df_ranking['sinal_cluster'])

df_ranking = df_ranking.sort_values(
    ['n_sinais', 'sim_centroide_eng'], ascending=False).reset_index(drop=True)

print('Distribuição de "geral" por nº de sinais:')
print(df_ranking['n_sinais'].value_counts().sort_index().to_string())
print(f'\nCandidatos com ≥2 sinais: {(df_ranking["n_sinais"] >= 2).sum()}')
print(f'Candidatos com 3 sinais (alta confiança): {(df_ranking["n_sinais"] == 3).sum()}')

df_ranking.head(30)[['numeroControlePNCP', 'text', 'sim_centroide_eng',
                       'pct_vizinhos_eng_obras', 'cluster_sbert',
                       'n_sinais']]

### 9.5 Validação semântica dos top suspeitos pelo LLM

O LLM olha o objeto e dá veredicto sobre a CLASSE CORRETA (engenharia/obras/geral)
SEM julgar rito legal — apenas a rotulação. Útil porque elimina falsos positivos
óbvios dos métodos vetoriais (ex.: contratos de "manutenção de ar condicionado
de servidor" que ficam perto de engenharia mas não são).

In [ ]:
import json, re

SYSTEM_ROTULACAO = """Você é analista de contratações públicas brasileiras
(Lei 14.133/2021). Sua tarefa é APENAS classificar o TIPO do contrato pelo
objeto, NÃO julgar se houve violação legal ou se seguiu rito formal.

Decida entre três classes:
- "engenharia": serviço técnico que tecnicamente exige profissional CREA/CAU
  (manutenção elétrica predial, instalação hidráulica, manutenção estrutural,
  projeto/laudo técnico, drenagem, etc.).
- "obras": construção, reforma estrutural, pavimentação asfáltica, demolição,
  ampliação física de edificação.
- "geral": serviço comum sem componente técnico de engenharia (limpeza,
  vigilância, alimentação, fornecimento de bens, transporte, manutenção de
  equipamentos não-prediais).

Responda APENAS no JSON:
{"classe_correta": "engenharia"|"obras"|"geral",
 "confianca": 0.0-1.0,
 "motivo": "frase curta (até 20 palavras)"}"""

def _parse_json_resposta(txt):
    if not txt: return None
    m = re.search(r"\{[^{}]*\}", txt, flags=re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

# Top-20 do ranking combinado → LLM dá veredito final
top = df_ranking.head(20).reset_index(drop=True)
veredictos = []
for i, row in top.iterrows():
    obj = str(row['text'])[:600]
    resp_txt = llm_task(MODELO_LLM, SYSTEM_ROTULACAO,
                          f"Objeto: {obj}")
    parsed = _parse_json_resposta(resp_txt) or {}
    veredictos.append({
        'numeroControlePNCP': row['numeroControlePNCP'],
        'objeto': obj[:120],
        'rotulo_original': row.get('rotulo'),
        'n_sinais': row['n_sinais'],
        'llm_classe': parsed.get('classe_correta', '?'),
        'llm_confianca': float(parsed.get('confianca', 0) or 0),
        'llm_motivo': parsed.get('motivo', '')[:200],
    })
    if (i + 1) % 5 == 0:
        print(f'  {i+1}/{len(top)}')

df_veredictos = pd.DataFrame(veredictos)
# Confirmados: LLM diz engenharia/obras com confiança ≥ 0.6
df_veredictos['confirmado_subenq'] = (
    df_veredictos['llm_classe'].isin(['engenharia', 'obras']) &
    (df_veredictos['llm_confianca'] >= 0.6)
)
n_conf = int(df_veredictos['confirmado_subenq'].sum())
print(f'\nLLM confirmou {n_conf}/{len(df_veredictos)} dos top suspeitos como '
      f'subenquadramento (engenharia/obras vestidos de geral).')
df_veredictos

## 10. Agente LLM com ferramentas (sem LangChain)

Agente minimalista: o Gemini decide qual ferramenta chamar respondendo em
JSON, executamos, devolvemos o resultado e repetimos até ele responder.

**Ferramentas disponíveis:**
- `busca_semantica_grafo(query)` — contratos similares a um texto livre.
- `listar_top_suspeitos(n)` — top-N do ranking consolidado da seção 9.

Sem LangChain porque a API muda de release para release e quebra. Esta versão
depende apenas de `google.generativeai` + `json` + `re`.

In [ ]:
"""
Agente LLM minimalista — sem LangChain.

Loop manual: prompt → Gemini → parse JSON → executa tool ou responde.
Estável porque depende só de google.generativeai + json/regex (vs. LangChain
que muda a API entre versões e quebra com frequência).
"""
import json, re

def busca_semantica_grafo(query=''):
    if not query:
        return 'Por favor, forneça uma consulta.'
    q_emb = sbert.encode([query], normalize_embeddings=True)[0]
    sims = cosine_similarity([q_emb], X_emb)[0]
    idx = int(np.argmax(sims))
    viz = list(G_emb.neighbors(idx)) + [idx]
    textos = []
    for v in viz[:5]:
        if v < len(df_amostra):
            row = df_amostra.iloc[v]
            textos.append(f"[{row.get('rotulo', '?')}] "
                           f"{row.get('numeroControlePNCP', '?')}: "
                           f"{str(row['text'])[:300]}")
    return (f'Contratos similares a "{query}":\n\n'
            + '\n\n===\n\n'.join(textos)
            + '\n\n=== Fim ===')

def listar_top_suspeitos(n=10):
    """Top-N do ranking final consolidado (3 sinais)."""
    if 'df_ranking' not in globals():
        return '(rode a seção 9 primeiro — falta o df_ranking)'
    sub = df_ranking.head(int(n))
    return '\n'.join(
        f"- [{r['n_sinais']} sinais] {r['numeroControlePNCP']}: "
        f"{str(r['text'])[:120]}"
        for _, r in sub.iterrows()
    )

_TOOLS = {
    'busca_semantica_grafo': busca_semantica_grafo,
    'listar_top_suspeitos': listar_top_suspeitos,
}

SYSTEM_AGENTE = """Você é um agente analítico para auditoria de contratos
públicos PNCP. Tem acesso a estas ferramentas:

- busca_semantica_grafo(query: str): busca contratos similares a um texto.
- listar_top_suspeitos(n: int): lista os top-N suspeitos do ranking final.

Quando precisar de dados, responda APENAS no JSON:
{"acao": "<nome_ferramenta>", "argumentos": {...}}

Quando tiver dados suficientes, responda APENAS no JSON:
{"acao": "responder", "resposta": "<texto natural com a conclusão>"}

Não escreva nada fora do JSON."""

def _parse_acao(txt):
    if not txt: return None
    # Pega o último bloco JSON da resposta
    matches = re.findall(r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", txt, flags=re.DOTALL)
    for m in reversed(matches):
        try: return json.loads(m)
        except Exception: continue
    return None

def agente(pergunta, max_passos=4, modelo='gemini-1.5-flash'):
    """Loop: LLM → tool → LLM → ... → resposta."""
    contexto = f"Pergunta: {pergunta}"
    for passo in range(max_passos):
        resp_txt = llm_task(modelo, SYSTEM_AGENTE, contexto)
        parsed = _parse_acao(resp_txt) or {}
        acao = parsed.get('acao')
        if acao == 'responder':
            return parsed.get('resposta', '(sem resposta)')
        if acao in _TOOLS:
            args = parsed.get('argumentos', {})
            try:
                saida = _TOOLS[acao](**args)
            except Exception as e:
                saida = f'(erro na ferramenta: {e})'
            contexto += (f'\n\nUsei {acao}({args}).'
                          f'\nResultado:\n{saida}')
            print(f'[passo {passo+1}] {acao}({args})')
        else:
            return f'(LLM não pediu ferramenta nem respondeu: {resp_txt[:200]})'
    return '(agente atingiu limite de passos)'

In [ ]:
resposta = agente(
    'Liste os 5 principais contratos suspeitos de subenquadramento '
    'e descreva o padrão comum entre eles.'
)
print('\n--- RESPOSTA FINAL ---\n')
print(resposta)